<H3> Import Modules

In [42]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import load_model

print("Module Imported Successfully...")

Module Imported Successfully...


<h3> Labeled Smile Faces with 0 and Non-Smile with 1

In [43]:
SmileFolder = "smile"
NonSmileFolder = "non_smile"

data = []

if os.path.exists(SmileFolder):
    for f in os.listdir(SmileFolder):
        if f.lower().endswith(('.jpg')):
            FilePath = os.path.join(SmileFolder, f)
            data.append({"image_path": FilePath, "label": 0})

if os.path.exists(NonSmileFolder):
    for f in os.listdir(NonSmileFolder):
        if f.lower().endswith(('.jpg')):
            FilePath = os.path.join(NonSmileFolder, f)
            data.append({"image_path": FilePath, "label": 1})

df = pd.DataFrame(data)
df.to_csv("labeled_faces.csv", index=False)

print(f"Dataset successfully labeled! Total images processed: {len(df)}")
print("\n\nLabeled Smile Faces : ")
print(df.head())
print("\n\nLabeled Non-Smile Faces : ")
print(df.tail())

Dataset successfully labeled! Total images processed: 1202


Labeled Smile Faces : 
      image_path  label
0    smile\1.jpg      0
1   smile\10.jpg      0
2  smile\100.jpg      0
3  smile\101.jpg      0
4  smile\102.jpg      0


Labeled Non-Smile Faces : 
            image_path  label
1197  non_smile\95.jpg      1
1198  non_smile\96.jpg      1
1199  non_smile\97.jpg      1
1200  non_smile\98.jpg      1
1201  non_smile\99.jpg      1


<h3> Load Dataset

In [44]:
print("Loding Dataset...")
df = pd.read_csv("labeled_faces.csv")

Loding Dataset...


<h3> Helper Function - Useful to load actual image from path

In [45]:
def LoadProcess(image_paths):
    images = []
    
    for path in image_paths:
        try:
            img = load_img(path, target_size=(128,128))
            img_array = img_to_array(img) / 255.0 # Normalize Pixels Value
            images.append(img_array)
            
        except Exception as e:
            print(f"Skipping corrupt or missing file: {path}")
            
    return np.array(images)

<h3>Split Dataset (80:20)

In [46]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

X_train, y_train = LoadProcess(train_df['image_path'].values), train_df['label'].values
X_val, y_val = LoadProcess(val_df['image_path'].values), val_df['label'].values

<h3>Model Creation

In [47]:
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    
    layers.Dense(1, activation='sigmoid')
])

<h3> Model Compilation

In [48]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

<h3>Model Summary

In [49]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)                    │ (None, 126, 126, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_6 (MaxPooling2D)       │ (None, 63, 63, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_7 (Conv2D)                    │ (None, 61, 61, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_7 (MaxPooling2D)       │ (None, 30, 30, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_8 (Conv2D)                    │ (None, 28, 28, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_8 (MaxPooling2D)       │ (None, 14, 14, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_2 (Flatten)                  │ (None, 25088)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 128)                 │       3,211,392 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,304,769 (12.61 MB)

 Trainable params: 3,304,769 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

<h3>Model Training

In [50]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_val, y_val)
)

Epoch 1/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 19s 510ms/step - accuracy: 0.5182 - loss: 0.7044 - val_accuracy: 0.5187 - val_loss: 0.6541
Epoch 2/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 20s 498ms/step - accuracy: 0.7045 - loss: 0.5557 - val_accuracy: 0.8797 - val_loss: 0.2807
Epoch 3/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 16s 510ms/step - accuracy: 0.8710 - loss: 0.3412 - val_accuracy: 0.9129 - val_loss: 0.1930
Epoch 4/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 16s 514ms/step - accuracy: 0.9136 - loss: 0.2315 - val_accuracy: 0.9295 - val_loss: 0.1727
Epoch 5/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 20s 493ms/step - accuracy: 0.9344 - loss: 0.1709 - val_accuracy: 0.9129 - val_loss: 0.2104
Epoch 6/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 20s 488ms/step - accuracy: 0.9469 - loss: 0.1462 - val_accuracy: 0.9253 - val_loss: 0.1784
Epoch 7/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 21s 491ms/step - accuracy: 0.9584 - loss: 0.1131 - val_accuracy: 0.9378 - val_loss: 0.1520
Epoch 8/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 21s 492ms/step - accuracy: 0.9740 - loss: 0.0868 - val_accu

<h3> Save Trained Parameters to Disk

In [51]:
model.save("SmileFaceClassifier.h5")
print("Model saved as SmileFaceClassifier.h5!")

Model saved as SmileFaceClassifier.h5!


<h3>Load Model

In [52]:
model = load_model("SmileFaceClassifier.h5")

<h3>Predict Model

In [84]:
def PredictFace(image_path):
    try:
        img = load_img(image_path, target_size=(128,128))
        img_array = img_to_array(img) / 255.0
        
        # Expand dimensions because Keras expects batches: (1, 128, 128, 3)
        img_array = np.expand_dims(img_array, axis=0)
        prediction = model.predict(img_array)[0][0]
        
        # 0 = Smile, 1 = Non-Smile
        if prediction < 0.5:
            confidence = (1 - prediction) * 100
            print(f"Image: {image_path}")
            print(f"Result: SMILING (Confidence: {confidence:.2f}%)\n")
        else:
            confidence = prediction * 100
            print(f"Image: {image_path}")
            print(f"Result: NOT SMILING (Confidence: {confidence:.2f}%)\n")
            
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")

In [85]:
image = [f"{i}.jpg" for i in range(1, 51)]
print(f"Loaded Images : {image}\n\n")
for name in image:
    image_path = os.path.join("test", name)
    PredictFace(image_path)

Loaded Images : ['1.jpg', '2.jpg', '3.jpg', '4.jpg', '5.jpg', '6.jpg', '7.jpg', '8.jpg', '9.jpg', '10.jpg', '11.jpg', '12.jpg', '13.jpg', '14.jpg', '15.jpg', '16.jpg', '17.jpg', '18.jpg', '19.jpg', '20.jpg', '21.jpg', '22.jpg', '23.jpg', '24.jpg', '25.jpg', '26.jpg', '27.jpg', '28.jpg', '29.jpg', '30.jpg', '31.jpg', '32.jpg', '33.jpg', '34.jpg', '35.jpg', '36.jpg', '37.jpg', '38.jpg', '39.jpg', '40.jpg', '41.jpg', '42.jpg', '43.jpg', '44.jpg', '45.jpg', '46.jpg', '47.jpg', '48.jpg', '49.jpg', '50.jpg']


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
Image: test\1.jpg
Result: SMILING (Confidence: 99.90%)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 332ms/step
Image: test\2.jpg
Result: SMILING (Confidence: 99.99%)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
Image: test\3.jpg
Result: SMILING (Confidence: 65.07%)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
Image: test\4.jpg
Result: SMILING (Confidence: 81.12%)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
Image: test\5.jpg
Result: NOT SMILING (Confidence: 56.17%)

1/1 ━━━━━━━━━━━